# Athena emulator — boundaries

You're hardening an integration against this stack and need to know exactly
how it fails: which mistakes are rejected at submit, which fail the
execution at runtime, what "not found" costs, and which operations are out
of scope by design. Every boundary below is probed live and records the
measured wire shape.

Stack (`docker-compose.yml`, network `mlops_net`): `athena` (emulator) ·
`trino` (query engine) · `moto` (S3/Glue) · `jupyterlab` (this notebook).
Errors travel the canonical JSON-1.1 envelope — `{"__type", "message"}` plus
the `X-Amzn-Errortype` header — so botocore parses every one into
`ClientError`.

In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import time
import uuid

import boto3
from _evidence import persist, record, render
from _helpers import (
    client_error_of,
    delete_run_buckets,
    delete_run_catalogs,
    delete_run_databases,
    delete_workgroup_contents,
    error_code_status,
    error_detail,
    query_execution,
    run_workgroup_names,
)

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL, (
    athena.meta.endpoint_url
)
assert s3.meta.endpoint_url.rstrip("/") == EDGE_URL, s3.meta.endpoint_url
assert glue.meta.endpoint_url.rstrip("/") == EDGE_URL, glue.meta.endpoint_url

REGION = os.environ["AWS_DEFAULT_REGION"]
ACCOUNT_ID = "123456789012"  # accepted but not enforced by the emulator
RUN_SUFFIX = uuid.uuid4().hex[:8]
# Every resource this run creates carries the nb09- prefix (nb09_ for Glue
# databases, where '-' is invalid) so cleanup can discover leftovers — its
# own and any failed run's — instead of tracking them in a list.
ATHENA_RESOURCE_PREFIX = "nb09-"
DATABASE_PREFIX = "nb09_"

bucket = f"{ATHENA_RESOURCE_PREFIX}{uuid.uuid4().hex[:12]}"
database = f"{DATABASE_PREFIX}{RUN_SUFFIX}"
disabled_workgroup = f"{ATHENA_RESOURCE_PREFIX}dis-{RUN_SUFFIX}"
lambda_catalog = f"{ATHENA_RESOURCE_PREFIX}cat-{RUN_SUFFIX}"
output_location = f"s3://{bucket}/results/"

s3.create_bucket(Bucket=bucket)
glue.create_database(DatabaseInput={"Name": database})
print(f"run scope: bucket={bucket} database={database}")


def poll_terminal(query_id):
    """Poll GetQueryExecution until the execution leaves QUEUED/RUNNING."""
    while True:
        execution = query_execution(athena, query_id)
        if execution["Status"]["State"] not in {"QUEUED", "RUNNING"}:
            return execution
        time.sleep(0.3)

run scope: bucket=nb09-53204296539c database=nb09_55d59221


## Rejected before a query exists

Bad SQL and a missing results location never create an execution — the
submit path answers `InvalidRequestException` 400 before any
`QueryExecutionId` exists, exactly where real Athena puts them.

In [3]:
syntax_error = client_error_of(
    lambda: athena.start_query_execution(
        QueryString="SELEC 1",
        ResultConfiguration={"OutputLocation": output_location},
    )
)
assert syntax_error is not None
assert error_code_status(syntax_error) == ("InvalidRequestException", 400)
# wrangler sniffs this prefix (awswrangler/athena/_utils.py:888-898).
assert syntax_error.response["Error"]["Message"].startswith(
    "Exception parsing query"
)
record(
    "syntax error rejected at submit → InvalidRequestException 400",
    "PASS",
    error_detail(syntax_error),
)

missing_output_error = client_error_of(
    lambda: athena.start_query_execution(
        QueryString="SELECT 1", WorkGroup="primary"
    )
)
assert missing_output_error is not None
assert error_code_status(missing_output_error) == (
    "InvalidRequestException",
    400,
)
record(
    "missing OutputLocation on primary → InvalidRequestException 400",
    "PASS",
    error_detail(missing_output_error),
)

Evidence(feature='missing OutputLocation on primary → InvalidRequestException 400', status='PASS', detail='InvalidRequestException (400): OutputLocation is required: pass ResultConfiguration.OutputLocation or configure', latency_ms=None)

## Workgroup and catalog gates

Unknown names, a `DISABLED` workgroup, and a catalog that isn't Glue-backed
all reject `StartQueryExecution` at submit — the same 400 shape, before
Trino ever sees the statement.

In [4]:
unknown_workgroup_error = client_error_of(
    lambda: athena.start_query_execution(
        QueryString="SELECT 1",
        WorkGroup=f"{ATHENA_RESOURCE_PREFIX}no-such",
        ResultConfiguration={"OutputLocation": output_location},
    )
)
assert unknown_workgroup_error is not None
assert error_code_status(unknown_workgroup_error) == (
    "InvalidRequestException",
    400,
)
record(
    "unknown workgroup rejected at submit → 400",
    "PASS",
    error_detail(unknown_workgroup_error),
)

athena.create_work_group(Name=disabled_workgroup)
athena.update_work_group(WorkGroup=disabled_workgroup, State="DISABLED")
disabled_error = client_error_of(
    lambda: athena.start_query_execution(
        QueryString="SELECT 1",
        WorkGroup=disabled_workgroup,
        ResultConfiguration={"OutputLocation": output_location},
    )
)
assert disabled_error is not None
assert error_code_status(disabled_error) == ("InvalidRequestException", 400)
record(
    "disabled workgroup rejects StartQueryExecution → 400",
    "PASS",
    error_detail(disabled_error),
)

unknown_catalog_error = client_error_of(
    lambda: athena.start_query_execution(
        QueryString="SELECT 1",
        QueryExecutionContext={
            "Catalog": f"{ATHENA_RESOURCE_PREFIX}no-such-cat"
        },
        ResultConfiguration={"OutputLocation": output_location},
    )
)
assert unknown_catalog_error is not None
assert error_code_status(unknown_catalog_error) == (
    "InvalidRequestException",
    400,
)
record(
    "unknown QueryExecutionContext.Catalog → 400",
    "PASS",
    error_detail(unknown_catalog_error),
)

athena.create_data_catalog(
    Name=lambda_catalog,
    Type="LAMBDA",
    Description="nb09 boundary probe",
    Parameters={
        "function": f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:nb09"
    },
)
non_glue_error = client_error_of(
    lambda: athena.start_query_execution(
        QueryString="SELECT 1",
        QueryExecutionContext={"Catalog": lambda_catalog},
        ResultConfiguration={"OutputLocation": output_location},
    )
)
assert non_glue_error is not None
assert error_code_status(non_glue_error) == ("InvalidRequestException", 400)
record(
    "non-GLUE catalog rejects query execution → 400",
    "PASS",
    error_detail(non_glue_error),
)

Evidence(feature='non-GLUE catalog rejects query execution → 400', status='PASS', detail='InvalidRequestException (400): DataCatalog nb09-cat-55d59221 is of type LAMBDA; only GLUE catalogs execute quer', latency_ms=None)

## Analysis failures land on the execution

A statement that parses but fails Trino analysis isn't a submit error — the
execution goes `FAILED` and `Status.StateChangeReason` carries the engine's
reason verbatim; its `GetQueryResults` answers an empty `Rows`.

In [5]:
started = time.monotonic()
missing_table_id = athena.start_query_execution(
    QueryString=f"SELECT * FROM {database}.missing_table",
    QueryExecutionContext={"Database": database},
    ResultConfiguration={"OutputLocation": output_location},
)["QueryExecutionId"]
missing_table_execution = poll_terminal(missing_table_id)
missing_table_ms = (time.monotonic() - started) * 1000
assert missing_table_execution["Status"]["State"] == "FAILED"
missing_table_reason = missing_table_execution["Status"]["StateChangeReason"]
assert "does not exist" in missing_table_reason
record(
    "missing-table SELECT → FAILED with Trino's reason verbatim",
    "PASS",
    f"{missing_table_id} FAILED: {missing_table_reason[:80]}",
    latency_ms=missing_table_ms,
)

started = time.monotonic()
duplicate_column_id = athena.start_query_execution(
    QueryString=(
        f"CREATE TABLE {database}.dup_{RUN_SUFFIX} AS SELECT 1 AS a, 2 AS a"
    ),
    QueryExecutionContext={"Database": database},
    ResultConfiguration={"OutputLocation": output_location},
)["QueryExecutionId"]
duplicate_column_execution = poll_terminal(duplicate_column_id)
duplicate_column_ms = (time.monotonic() - started) * 1000
assert duplicate_column_execution["Status"]["State"] == "FAILED"
duplicate_column_reason = duplicate_column_execution["Status"][
    "StateChangeReason"
]
assert "specified more than once" in duplicate_column_reason
record(
    "duplicate-column CTAS → FAILED with Trino's reason verbatim",
    "PASS",
    f"{duplicate_column_id} FAILED: {duplicate_column_reason[:80]}",
    latency_ms=duplicate_column_ms,
)

failed_rows = athena.get_query_results(QueryExecutionId=duplicate_column_id)[
    "ResultSet"
]["Rows"]
assert failed_rows == []
record(
    "GetQueryResults on a FAILED execution → empty Rows",
    "PASS",
    f"{duplicate_column_id} answered Rows=[]",
)

Evidence(feature='GetQueryResults on a FAILED execution → empty Rows', status='PASS', detail='6edfcb63-8ac9-4238-9961-3b06ecbadb2e answered Rows=[]', latency_ms=None)

## Results before the finish line

`GetQueryResults` on an execution that hasn't reached a terminal state is
the same `InvalidRequestException` 400 real Athena sends — the message names
the live state. `StopQueryExecution` unwinds the probe query.

In [6]:
slow_query_id = athena.start_query_execution(
    QueryString=(
        "SELECT COUNT(DISTINCT x) FROM UNNEST(SEQUENCE(1, 200000000)) AS t(x)"
    ),
    ResultConfiguration={"OutputLocation": output_location},
)["QueryExecutionId"]

early_results_error = client_error_of(
    lambda: athena.get_query_results(QueryExecutionId=slow_query_id)
)
assert early_results_error is not None
assert error_code_status(early_results_error) == (
    "InvalidRequestException",
    400,
)
assert (
    "has not yet finished" in early_results_error.response["Error"]["Message"]
)
record(
    "GetQueryResults on a QUEUED/RUNNING execution → 400",
    "PASS",
    error_detail(early_results_error),
)

athena.stop_query_execution(QueryExecutionId=slow_query_id)
stopped_execution = poll_terminal(slow_query_id)
assert stopped_execution["Status"]["State"] in {"CANCELLED", "SUCCEEDED"}
record(
    "in-flight execution unwound via StopQueryExecution",
    "PASS",
    f"{slow_query_id} → {stopped_execution['Status']['State']}",
)

Evidence(feature='in-flight execution unwound via StopQueryExecution', status='PASS', detail='c9dd4011-f2b3-4509-a1ee-5df57f64343f → CANCELLED', latency_ms=None)

## Not found — 404 only where the model declares it

`ResourceNotFoundException` exists only where service-2.json declares it —
prepared-statement and tag ops. Every other missing read answers
`InvalidRequestException` 400, and a name that fails the model's pattern
never reaches the lookup.

In [7]:
pattern_error = client_error_of(
    lambda: athena.get_prepared_statement(
        StatementName="nb09-bad-name", WorkGroup="primary"
    )
)
assert pattern_error is not None
assert error_code_status(pattern_error) == ("InvalidRequestException", 400)
record(
    "StatementName pattern violation → 400 before the lookup",
    "PASS",
    error_detail(pattern_error),
)

missing_statement_error = client_error_of(
    lambda: athena.get_prepared_statement(
        StatementName=f"nb09_nosuch_{RUN_SUFFIX}", WorkGroup="primary"
    )
)
assert missing_statement_error is not None
assert error_code_status(missing_statement_error) == (
    "ResourceNotFoundException",
    404,
)
record(
    "missing prepared statement → ResourceNotFoundException 404",
    "PASS",
    error_detail(missing_statement_error),
)

missing_execution_error = client_error_of(
    lambda: athena.get_query_execution(
        QueryExecutionId="00000000-0000-0000-0000-000000000000"
    )
)
assert missing_execution_error is not None
assert error_code_status(missing_execution_error) == (
    "InvalidRequestException",
    400,
)
record(
    "unknown QueryExecutionId → InvalidRequestException 400",
    "PASS",
    error_detail(missing_execution_error),
)

Evidence(feature='unknown QueryExecutionId → InvalidRequestException 400', status='PASS', detail='InvalidRequestException (400): QueryExecution 00000000-0000-0000-0000-000000000000 does not exist', latency_ms=None)

## Out of scope by design

Studio-era session/notebook/calculation and capacity ops are present in
dispatch but unimplemented (ADR-0004): each answers a shaped
`InvalidRequestException` 400 naming the operation — never a raw 404/501
that would break botocore error parsing.

In [8]:
out_of_scope_calls = {
    "ListSessions": lambda: athena.list_sessions(WorkGroup="primary"),
    "StartCalculationExecution": (
        lambda: athena.start_calculation_execution(
            SessionId=f"nb09-{RUN_SUFFIX}"
        )
    ),
    "ListApplicationDPUSizes": lambda: athena.list_application_dpu_sizes(),
    "CreateNotebook": lambda: athena.create_notebook(
        WorkGroup="primary", Name=f"nb09-{RUN_SUFFIX}"
    ),
}
for operation_name, call in out_of_scope_calls.items():
    outcome = client_error_of(call)
    assert outcome is not None, operation_name
    assert error_code_status(outcome) == ("InvalidRequestException", 400)
    assert operation_name in outcome.response["Error"]["Message"]
record(
    "out-of-scope ops (session/calculation/DPU/notebook) → shaped 400",
    "PASS",
    "; ".join(
        f"{name} → InvalidRequestException" for name in out_of_scope_calls
    ),
)

Evidence(feature='out-of-scope ops (session/calculation/DPU/notebook) → shaped 400', status='PASS', detail='ListSessions → InvalidRequestException; StartCalculationExecution → InvalidRequestException; ListApplicationDPUSizes → InvalidRequestException; CreateNotebook → InvalidRequestException', latency_ms=None)

## Capabilities demonstrated

Every feature above recorded a measured row. The matrix persists to
`parity/09_boundaries.md`; `PARITY.md` at the project root is regenerated
from all fragments.

In [9]:
print(render())
persist("09_boundaries")

| feature | status | detail | latency_ms |
|---|---|---|---|
| syntax error rejected at submit → InvalidRequestException 400 | PASS | InvalidRequestException (400): Exception parsing query: line 1:1: mismatched input 'SELEC'. Expecting: 'ALTER', |  |
| missing OutputLocation on primary → InvalidRequestException 400 | PASS | InvalidRequestException (400): OutputLocation is required: pass ResultConfiguration.OutputLocation or configure |  |
| unknown workgroup rejected at submit → 400 | PASS | InvalidRequestException (400): WorkGroup nb09-no-such does not exist |  |
| disabled workgroup rejects StartQueryExecution → 400 | PASS | InvalidRequestException (400): WorkGroup nb09-dis-55d59221 is disabled |  |
| unknown QueryExecutionContext.Catalog → 400 | PASS | InvalidRequestException (400): DataCatalog nb09-no-such-cat does not exist |  |
| non-GLUE catalog rejects query execution → 400 | PASS | InvalidRequestException (400): DataCatalog nb09-cat-55d59221 is of type LAMBDA; only GLUE catalo

PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/09_boundaries.md')

## Clean up

The run's workgroup, catalog, bucket, and Glue database are throwaway —
delete them so the stack is left as you found it; the prefix sweep also
collects leftovers a crashed run may have abandoned.

In [10]:
for workgroup_name in run_workgroup_names(athena, ATHENA_RESOURCE_PREFIX):
    delete_workgroup_contents(athena, workgroup_name)
    athena.delete_work_group(WorkGroup=workgroup_name)
delete_run_catalogs(athena, ATHENA_RESOURCE_PREFIX)
delete_run_databases(glue, DATABASE_PREFIX)
delete_run_buckets(s3, ATHENA_RESOURCE_PREFIX)
print(f"cleaned run scope: {ATHENA_RESOURCE_PREFIX}* / {DATABASE_PREFIX}*")

cleaned run scope: nb09-* / nb09_*
